# Capstone: weekly habit finding

**Weekly habit finding with a model** (`scripts/eval_weekly_habits.py`, `habits/weekly.py`): every week,
CatBoost predicts from the last 3 weeks which slots will be habits next week, per appliance (so one slot
can hold several habits), and keeps or drops existing ones (two thresholds). Its inputs include the other
appliances in four ways (same slot, order, together or apart, room groups), tested as an ablation. Against
counting once (today's system), counting over the last 3 weeks, and the last 3 weeks' rate; each home is
predicted by a model trained on the other homes. Then SHAP by input group.

**Settings:** Accelerator = **None** (CPU), Internet = **On** (CASAS from Zenodo), dataset `capstone-bundle`
(latest version) under Input. Save Version > Save & Run All (Commit).

**A new notebook copies none of these settings.**

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
import catboost, shap, warnings
warnings.filterwarnings('ignore', message='LightGBM binary classifier')
print('catboost', catboost.__version__, '| shap', shap.__version__)
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
notify('Weekly habits: started', 'CASAS download, then the runs', 'default', 'rocket')

In [ ]:
!mkdir -p /kaggle/temp && wget -q -c -O /kaggle/temp/casas.zip "https://zenodo.org/records/15708568/files/labeled_data.zip?download=1"
!ls -l /kaggle/temp/casas.zip   # should be 236037656 bytes
ok = run_step('CASAS prepare', 'CAPSTONE_DATASET=casas python -u scripts/prepare_casas.py /kaggle/temp/casas.zip',
              'logs/casas_prepare.log', summary_lines=3)
casas_ok = ok  # without CASAS, its steps are skipped (REFIT and UK-DALE still run)

In [ ]:
DATASETS = ['refit', 'ukdale', 'casas']   # to rerun only CASAS: ['casas']
ok &= run_step('Tests', 'python -m pytest -q tests/test_weekly.py', 'logs/tests.log', summary_lines=2)
for ds in DATASETS:
    if ds == 'casas' and not casas_ok:
        continue
    ok &= run_step(f'{ds} weekly habits', 'python -u scripts/eval_weekly_habits.py',
                   f'logs/{ds}_weekly_habits.log', env={'CAPSTONE_DATASET': ds}, summary_lines=40)

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/weekly_habits.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in [*pathlib.Path('data/results').rglob('weekly_habits*.csv'), *pathlib.Path('logs').rglob('*.log')]:
        zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/weekly_habits.zip').stat().st_size / 1e6, 2), 'MB')
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
notify('Weekly habits: FINISHED' if ok else 'Weekly habits: finished WITH ERRORS',
       'weekly_habits.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')